# Molecular Dynamics: Does Aspirin Stay Put in COX-1?
Docking gives a single frozen snapshot. Here we simulate the **aspirin–COX-1 complex in water** with OpenMM and watch whether aspirin stays in the binding channel and keeps contact with the key residues (Arg120, Tyr355, Ser530).

**Plan**
1. Install tools (takes a few minutes, one kernel restart)
2. Prepare protein, ligand, and a water box
3. Minimize, heat, and run production MD
4. Analyze stability (RMSD, contacts) and animate

**Use a GPU runtime:** Runtime → Change runtime type → T4 GPU.

## 1. Install
Run the next cell. **The kernel will restart automatically**; then continue with the cell after it.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
import condacolab
condacolab.check()
!mamba install -q -y -c conda-forge openmm openmmforcefields openff-toolkit ambertools pdbfixer mdtraj rdkit py3Dmol pandas matplotlib requests
!nvidia-smi -L

## 2. Choose the ligand pose
Upload `aspirin_best.pdb` from the docking notebook (left sidebar → Files in that session, right-click → Download). If you skip the upload, the crystal ibuprofen pose is used instead.

In [ ]:
import os
if not os.path.exists("aspirin_best.pdb"):
    try:
        from google.colab import files
        files.upload()
    except Exception as e:
        print("No upload:", e)
print("Aspirin pose found:", os.path.exists("aspirin_best.pdb"))

## 3. Load COX-1 and the ligand

In [ ]:
import requests, numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

pdb = requests.get("https://files.rcsb.org/download/1EQG.pdb", timeout=30).text
lines = pdb.splitlines()
ibp_all = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == "IBP" and l[16] in " A"]
chain = ibp_all[0][21]
ibp = [l for l in ibp_all if l[21] == chain]
protein = [l for l in lines if l.startswith("ATOM") and l[21] == chain and l[16] in " A"]
open("receptor.pdb", "w").write("\n".join(protein) + "\nEND\n")

# Carboxylic acids are deprotonated at body pH, so we use the anion forms
if os.path.exists("aspirin_best.pdb"):
    LIG_NAME, LIG_SMILES = "aspirin", "CC(=O)Oc1ccccc1C(=O)[O-]"
    lig_block = open("aspirin_best.pdb").read()
else:
    LIG_NAME, LIG_SMILES = "ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)[O-]"
    lig_block = "\n".join(ibp)

template = Chem.MolFromSmiles(LIG_SMILES)
pose = Chem.MolFromPDBBlock(lig_block, removeHs=True)
pose = AllChem.AssignBondOrdersFromTemplate(template, pose)
pose = Chem.AddHs(pose, addCoords=True)
Chem.AssignStereochemistryFrom3D(pose)
print("Ligand:", LIG_NAME, "| atoms with H:", pose.GetNumAtoms(), "| charge:", Chem.GetFormalCharge(pose))

## 4. Prepare the protein
PDBFixer adds missing atoms and hydrogens (pH 7). Heme, sugars, and crystal waters are removed. Short internal gaps are rebuilt crudely; minimization relaxes them.

In [ ]:
from pdbfixer import PDBFixer
import openmm as mm
import openmm.app as app
from openmm import unit

fixer = PDBFixer(filename="receptor.pdb")
fixer.findNonstandardResidues(); fixer.replaceNonstandardResidues()
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues()
chains = list(fixer.topology.chains())
for key in list(fixer.missingResidues.keys()):          # drop missing residues at chain ends only
    if key[1] == 0 or key[1] == len(list(chains[key[0]].residues())):
        del fixer.missingResidues[key]
fixer.findMissingAtoms(); fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)
n_prot_res = fixer.topology.getNumResidues()
print("Protein residues:", n_prot_res)

## 5. Parametrize the ligand, add water and ions
Protein: Amber14. Ligand: GAFF2 with AM1-BCC charges. Water: TIP3P-FB, 0.15 M salt.

In [ ]:
from openff.toolkit import Molecule
from openff.units.openmm import to_openmm
from openmmforcefields.generators import GAFFTemplateGenerator

off_mol = Molecule.from_rdkit(pose, allow_undefined_stereo=True)
gaff = GAFFTemplateGenerator(molecules=off_mol, forcefield="gaff-2.11")
ff = app.ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
ff.registerTemplateGenerator(gaff.generator)

modeller = app.Modeller(fixer.topology, fixer.positions)
lig_top = off_mol.to_topology().to_openmm()
for r in lig_top.residues():
    r.name = "LIG"
modeller.add(lig_top, to_openmm(off_mol.conformers[0]))
modeller.addSolvent(ff, model="tip3p", padding=0.9 * unit.nanometer,
                    ionicStrength=0.15 * unit.molar, neutralize=True)
print("Total atoms:", modeller.topology.getNumAtoms())

with open("system.pdb", "w") as f:
    app.PDBFile.writeFile(modeller.topology, modeller.positions, f)

## 6. Build the simulation
- 4 fs time step (hydrogen mass repartitioning)
- Backbone and ligand are gently restrained during heating, then released
- Pressure kept at 1 bar

In [ ]:
system = ff.createSystem(modeller.topology, nonbondedMethod=app.PME,
                         nonbondedCutoff=1.0 * unit.nanometer,
                         constraints=app.HBonds, hydrogenMass=1.5 * unit.amu)
system.addForce(mm.MonteCarloBarostat(1 * unit.bar, 300 * unit.kelvin))

# Position restraints on protein CA atoms and ligand heavy atoms
restraint = mm.CustomExternalForce("k*periodicdistance(x,y,z,x0,y0,z0)^2")
restraint.addGlobalParameter("k", 1000.0)   # kJ/mol/nm^2
for p in ("x0", "y0", "z0"):
    restraint.addPerParticleParameter(p)
pos_nm = modeller.positions.value_in_unit(unit.nanometer)
for a in modeller.topology.atoms():
    is_prot_ca = a.name == "CA" and a.residue.index < n_prot_res
    is_lig_heavy = a.residue.name == "LIG" and a.element.symbol != "H"
    if is_prot_ca or is_lig_heavy:
        restraint.addParticle(a.index, list(pos_nm[a.index]))
system.addForce(restraint)

integrator = mm.LangevinMiddleIntegrator(50 * unit.kelvin, 1 / unit.picosecond, 0.004 * unit.picoseconds)
platform = max((mm.Platform.getPlatform(i) for i in range(mm.Platform.getNumPlatforms())),
               key=lambda p: p.getSpeed())
props = {"Precision": "mixed"} if platform.getName() in ("CUDA", "OpenCL") else {}
sim = app.Simulation(modeller.topology, system, integrator, platform, props)
sim.context.setPositions(modeller.positions)
print("Platform:", platform.getName())
if platform.getName() not in ("CUDA", "OpenCL"):
    print("WARNING: no GPU found. Switch to a GPU runtime or the run will be very slow.")

## 7. Minimize and heat (50 K → 300 K)

In [ ]:
sim.minimizeEnergy(maxIterations=2000)
sim.context.setVelocitiesToTemperature(50 * unit.kelvin)
for T in range(50, 301, 50):
    integrator.setTemperature(T * unit.kelvin)
    sim.step(5000)                      # 20 ps per stage
    print(f"{T} K done")
sim.context.setParameter("k", 0.0)      # release restraints
print("Restraints released")

## 8. Production run
Default is **500 ps**, about 10-15 minutes on a T4. Longer is better (try 2000 ps if time allows). Frames are saved every 10 ps.

In [ ]:
PROD_PS = 500 if platform.getName() in ("CUDA", "OpenCL") else 10
REPORT_PS = 10
steps = int(PROD_PS / 0.004)
interval = int(REPORT_PS / 0.004)

sim.reporters.append(app.DCDReporter("traj.dcd", interval))
sim.reporters.append(app.StateDataReporter("md_log.csv", interval, step=True, time=True,
                     potentialEnergy=True, temperature=True, density=True, speed=True))
sim.step(steps)
print("Production finished:", PROD_PS, "ps")

## 9. Check the simulation was healthy
Temperature should hover near 300 K and density near 1.0 g/mL.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv("md_log.csv")
log.columns = ["step", "time_ps", "potential", "temp", "density", "speed"]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(log.time_ps, log.temp); ax[0].set_title("Temperature (K)"); ax[0].set_xlabel("ps")
ax[1].plot(log.time_ps, log.density); ax[1].set_title("Density (g/mL)"); ax[1].set_xlabel("ps")
plt.tight_layout(); plt.show()

## 10. Analysis: is aspirin stable in the pocket?

In [ ]:
import mdtraj as md
traj = md.load("traj.dcd", top="system.pdb")
traj.image_molecules(inplace=True)

bb = traj.topology.select("protein and backbone")
lig = traj.topology.select("resname LIG and not element H")
ref = traj[0]
traj.superpose(ref, atom_indices=bb)

time_ns = np.arange(traj.n_frames) * REPORT_PS / 1000
prot_rmsd = md.rmsd(traj, ref, atom_indices=bb) * 10                       # Å
lig_rmsd = np.sqrt(((traj.xyz[:, lig] - ref.xyz[:, lig]) ** 2).sum(axis=2).mean(axis=1)) * 10

KEY = {"ARG120", "TYR355", "SER530", "TYR385"}
key_idx = {f"{r.name}{r.id}": r.index for r in fixer.topology.residues() if f"{r.name}{r.id}" in KEY}

def min_dist(res_index):
    atoms = [a.index for a in traj.topology.residue(res_index).atoms if a.element.symbol != "H"]
    pairs = np.array([(i, j) for i in atoms for j in lig])
    return md.compute_distances(traj, pairs).min(axis=1) * 10

dists = {name: min_dist(i) for name, i in key_idx.items()}
print("Key residues found:", list(key_idx))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].plot(time_ns, prot_rmsd, color="#4C78A8"); ax[0].set_title("Protein backbone RMSD (Å)")
ax[1].plot(time_ns, lig_rmsd, color="#E45756"); ax[1].set_title(f"{LIG_NAME.title()} RMSD (Å)")
for name, d in dists.items():
    ax[2].plot(time_ns, d, label=name)
ax[2].axhline(4, ls="--", color="gray"); ax[2].set_title("Min distance to key residues (Å)"); ax[2].legend(fontsize=7)
for a in ax: a.set_xlabel("ns")
plt.tight_layout(); plt.savefig("md_analysis.png", dpi=150); plt.show()

print(f"Mean {LIG_NAME} RMSD: {lig_rmsd.mean():.2f} Å | max: {lig_rmsd.max():.2f} Å")
print("\n| Residue | % of frames in contact (<4 Å) |\n|---|---|")
for name, d in dists.items():
    print(f"| {name} | {100 * (d < 4).mean():.0f}% |")

**How to read it**
- Protein RMSD leveling off means the enzyme is stable.
- Ligand RMSD staying low (a few Å) and contacts staying below the dashed 4 Å line means the ligand stays bound. A big jump means it moved or left the pocket.
- Short runs cannot prove binding. They can only show whether the docked pose is *plausible* or *quickly falls apart*.

## 11. Animate the binding pocket

In [ ]:
import py3Dmol
nb = md.compute_neighbors(traj[0], 1.0, lig, haystack_indices=traj.topology.select("protein"))[0]
pocket_res = {traj.topology.atom(i).residue.index for i in nb}
keep = [a.index for a in traj.topology.atoms
        if a.residue.index in pocket_res or a.residue.name == "LIG"]
sub = traj.atom_slice(keep)[::2]
sub.save_pdb("pocket_traj.pdb")

view = py3Dmol.view(width=600, height=480)
view.addModelsAsFrames(open("pocket_traj.pdb").read(), "pdb")
view.setStyle({}, {"stick": {"radius": 0.08, "colorscheme": "cyanCarbon"}})
view.setStyle({"resn": "LIG"}, {"stick": {"radius": 0.22, "colorscheme": "greenCarbon"}})
view.zoomTo({"resn": "LIG"})
view.animate({"loop": "forward", "reps": 0, "interval": 120})
view.show()

## Limitations
- Short simulation (hundreds of picoseconds), not enough for unbinding events or free energies.
- Heme and sugars were removed; one monomer only.
- Aspirin acetylates Ser530 covalently, which classical MD cannot model.
- Single run: repeat with different random seeds for confidence.

## Next steps
- Run longer (2-10 ns) and repeat 3 times
- Calculate binding free energy (MM-GBSA)
- Compare with COX-2